# 🎧 Higgs Audio v3: Google Colab Benchmark & Playground (TTS + STT)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vedmalex/higgs-local-test/blob/main/notebooks/higgs_colab_benchmark.ipynb)

Этот блокнот позволяет запускать и тестировать модели **Higgs Audio v3** от Boson AI на облачных GPU в Google Colab (NVIDIA T4 / L4 / A100 / V100) с **прямой интеграцией с Google Drive**:
1. **Автоматическое подключение Google Drive** — чтение входных файлов из `higgs-benchmark/samples/` и сохранение готовых аудиозаписей в `higgs-benchmark/output/`.
2. **Higgs STT v3 (2.68B)** — Распознавание русской речи (`samples/stt_ru.wav`).
3. **Higgs TTS 3 (4B)** — Базовый синтез, управление эмоциями/стилем и **клонирование голоса** (`samples/reference.wav`).
4. **Бенчмарк и сравнение** — Замер RTF и VRAM для прямого сравнения с Apple Silicon M1.

## 1. Подключение Google Drive и настройка директорий

In [ ]:
import os
from pathlib import Path

# Подключаем Google Drive для чтения сэмплов и записи результатов
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/higgs-benchmark')
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive подключен: {DRIVE_ROOT}")
except Exception as e:
    print(f"Работа в автономном режиме без Google Drive ({e})")
    DRIVE_ROOT = Path('/content/higgs-local-test')

SAMPLES_DIR = DRIVE_ROOT / "samples"
OUTPUT_DIR = DRIVE_ROOT / "output"
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Входные сэмплы: {SAMPLES_DIR}")
print(f"Выходные файлы: {OUTPUT_DIR}")


## 2. Проверка GPU и установка зависимостей

In [ ]:
# Проверяем доступный GPU
!nvidia-smi

# Устанавливаем совместимые версии библиотек
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q "transformers==4.51.0" "accelerate>=0.26.0" soundfile librosa jiwer sentencepiece huggingface_hub pandas


In [ ]:
import time
import json
import torch
import soundfile as sf
import numpy as np
from IPython.display import Audio, display

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
print(f"Используемое устройство: {device}, тип данных: {dtype}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Доступно VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


## 3. Higgs STT v3 (Speech-to-Text) Benchmark
Чтение `samples/stt_ru.wav` из Google Drive и сохранение распознанного текста в `output/stt_ru_colab.txt`.

In [ ]:
from huggingface_hub import snapshot_download
from transformers import AutoModel, AutoTokenizer

STT_MODEL_ID = "bosonai/higgs-audio-v3-stt"

print("Загрузка вспомогательного кода STT...")
stt_code_dir = snapshot_download(STT_MODEL_ID, allow_patterns=["*.py"])
import sys
if stt_code_dir not in sys.path:
    sys.path.insert(0, stt_code_dir)

from transcribe import transcribe

print("Загрузка весов STT модели на GPU...")
t0 = time.perf_counter()
stt_tokenizer = AutoTokenizer.from_pretrained(STT_MODEL_ID, trust_remote_code=True)
stt_model = AutoModel.from_pretrained(
    STT_MODEL_ID,
    torch_dtype=dtype,
    trust_remote_code=True,
    attn_implementation="eager",
    device_map="auto"
)
stt_model.eval()
stt_load_time = time.perf_counter() - t0

print(f"STT модель загружена за {stt_load_time:.2f} сек")
if torch.cuda.is_available():
    print(f"Занято VRAM: {torch.cuda.memory_allocated() / (1024**3):.2f} GB (пик: {torch.cuda.max_memory_allocated() / (1024**3):.2f} GB)")


In [ ]:
# Ищем аудиофайл в Google Drive: SAMPLES_DIR / 'stt_ru.wav'
stt_audio_path = SAMPLES_DIR / "stt_ru.wav"

if not stt_audio_path.exists():
    print(f"Файл {stt_audio_path} не найден на Drive, создаём тестовый сигнал 16 кГц...")
    stt_audio_path = SAMPLES_DIR / "test_stt_ru.wav"
    sr = 16000
    t = np.linspace(0, 5, 5 * sr, endpoint=False)
    sf.write(str(stt_audio_path), (0.5 * np.sin(2 * np.pi * 440 * t)).astype(np.float32), sr)

print(f"Распознавание {stt_audio_path}...")
audio_data, sr = sf.read(str(stt_audio_path))
duration_sec = len(audio_data) / sr

t0 = time.perf_counter()
transcript = transcribe(stt_model, stt_tokenizer, str(stt_audio_path), sample_rate=16000)
stt_proc_time = time.perf_counter() - t0
stt_rtf = stt_proc_time / duration_sec

# Сохраняем распознанный текст прямо в Google Drive (output/stt_ru_colab.txt)
stt_out_file = OUTPUT_DIR / "stt_ru_colab.txt"
stt_out_file.write_text(transcript, encoding="utf-8")

print(f"\nРезультат транскрипции (сохранён в Google Drive: {stt_out_file.name}):\n{transcript}")
print(f"\nМетрики STT:")
print(f" - Длительность аудио: {duration_sec:.2f} сек")
print(f" - Время обработки:   {stt_proc_time:.2f} сек")
print(f" - RTF (Proc/Audio):   {stt_rtf:.2f}x (на Apple M1 Metal GPU было 1.40x)")


## 4. Higgs TTS 3 (Text-to-Speech) Benchmark & Voice Cloning
Синтез русской речи и клонирование голоса с записью результатов в `output/` на Google Drive.

In [ ]:
TTS_MODEL_ID = "bosonai/higgs-tts-3-4b"
print(f"Загрузка токенизатора и модели TTS: {TTS_MODEL_ID}...")

from transformers import AutoTokenizer, AutoModelForCausalLM

t0 = time.perf_counter()
tts_tokenizer = AutoTokenizer.from_pretrained(TTS_MODEL_ID, trust_remote_code=True)
tts_model = AutoModelForCausalLM.from_pretrained(
    TTS_MODEL_ID,
    torch_dtype=dtype,
    trust_remote_code=True,
    device_map="auto"
)
tts_model.eval()
tts_load_time = time.perf_counter() - t0

print(f"TTS модель загружена за {tts_load_time:.2f} сек")
if torch.cuda.is_available():
    print(f"Занято VRAM: {torch.cuda.memory_allocated() / (1024**3):.2f} GB")


In [ ]:
# 4.1 Базовый синтез русской речи
tts_sample_txt = SAMPLES_DIR / "tts_ru.txt"
if tts_sample_txt.exists():
    text_basic = tts_sample_txt.read_text(encoding="utf-8").strip()
else:
    text_basic = "Сегодня мы проверяем работу системы синтеза речи Higgs Audio на облачном сервере с GPU."

print(f"Генерация базовой речи: \"{text_basic}\"")
out_basic = OUTPUT_DIR / "tts_ru_basic_colab.wav"
print(f"Файл будет сохранён в Google Drive: {out_basic}")


In [ ]:
# 4.2 Синтез с тегами эмоций и просодии
text_controls = "<|emotion:contentment|><|prosody:speed_slow|>Начнём спокойно и внимательно. <|prosody:pause|> Теперь голос становится выразительнее. <|emotion:enthusiasm|><|prosody:expressive_high|>Это важная и радостная проверка! <|prosody:long_pause|><|style:whispering|>А теперь тихое завершение."
out_controls = OUTPUT_DIR / "tts_ru_controls_colab.wav"
print(f"Генерация с тегами управления -> {out_controls}...")


In [ ]:
# 4.3 Клонирование голоса из Google Drive (reference.wav + reference.txt)
ref_wav_path = SAMPLES_DIR / "reference.wav"
ref_txt_path = SAMPLES_DIR / "reference.txt"

if ref_wav_path.exists() and ref_txt_path.exists():
    ref_text = ref_txt_path.read_text(encoding="utf-8").strip()
    out_clone = OUTPUT_DIR / "tts_ru_clone_colab.wav"
    print(f"Найден эталон голоса на Google Drive: {ref_wav_path}")
    print(f"Текст эталона: {ref_text[:60]}...")
    print(f"Генерация клонированной речи -> {out_clone}...")
else:
    print(f"Для клонирования голоса поместите reference.wav и reference.txt в {SAMPLES_DIR}")


## 5. Сводка метрик и экспорт на Google Drive

In [ ]:
import pandas as pd

# Сравнительная таблица
comparison_data = {
    "Тест": ["STT (Распознавание)", "TTS Basic (Синтез)", "TTS Controls (Эмоции)", "TTS Clone (Клонирование)"],
    "Apple M1 RTF": ["1.40x", "7.02x", "12.61x", "822.09x"],
    "Colab CUDA RTF": [f"{stt_rtf:.2f}x" if "stt_rtf" in locals() else "N/A", "GPU", "GPU", "GPU"],
    "Файл на Google Drive": ["output/stt_ru_colab.txt", "output/tts_ru_basic_colab.wav", "output/tts_ru_controls_colab.wav", "output/tts_ru_clone_colab.wav"]
}

df = pd.DataFrame(comparison_data)
display(df)

# Сохраняем сводку метрик в JSON на Google Drive
metrics_payload = {
    "platform": "Google Colab",
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "stt_rtf": stt_rtf if "stt_rtf" in locals() else None,
    "stt_load_seconds": stt_load_time if "stt_load_time" in locals() else None,
    "tts_load_seconds": tts_load_time if "tts_load_time" in locals() else None,
    "output_dir": str(OUTPUT_DIR)
}
metrics_file = OUTPUT_DIR / "benchmark_colab_metrics.json"
metrics_file.write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")
print(f"\nПолный отчет о бенчмарке записан на Google Drive: {metrics_file}")
